# Analisie Exploratoria de Dados - Telco Customer Churn

## Importando as bibliotecas

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Configuracoes de visualizacao
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

## Carregamento

In [ ]:
df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

print(f"Shape: {df.shape}")
df.head()


Shape: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


### Observações iniciais — `df.head()`

- **7.043 clientes** e **33 colunas** no total.
- As primeiras linhas revelam clientes de **Los Angeles / California** — isso já levanta a suspeita de que `Country`, `State` e colunas de geolocalização podem ter baixa variância.
- `Total Charges` aparece como string (`108.15`, `151.65`) mas deveria ser numérico — anomalia a investigar.
- `Churn Label` e `Churn Value` carregam a mesma informação em formatos diferentes (`Yes/No` vs `1/0`).

## Inspecao de tipos e valores nulos:

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

### Observações — `df.info()`

| Achado | Detalhe |
|--------|---------|
| **`Total Charges` é `object`** | Deveria ser `float64`. Indica valores não numéricos que impedem a inferência automática do tipo. |
| **`Churn Reason` tem 5.174 nulos** | Apenas 1.869 registros preenchidos — exatamente os clientes que fizeram churn. Os nulos são estruturais, não sujeira. |
| **Sem outros nulos aparentes** | Todas as demais colunas mostram 7.043 non-null, o que é positivo para a qualidade geral do dataset. |

> **Atenção:** O `.info()` reporta `Total Charges` com 7.043 non-null, mas isso **não significa ausência de problemas** — strings vazias `""` são contadas como valores válidos pelo pandas.

## Resumo estatistico separado por tipo:

In [ ]:
print("=== Variaveis Numericas ===")
display(df.describe())

print("=== Variaveis CategÃ³ricas ===")
display(df.describe(include="object"))


=== Variaveis Numericas ===


,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,1.000000,100.000000,6500.000000


=== Variaveis CategÃ³ricas ===


,CustomerID,Country,State,City,Lat Long,Gender,Senior Citizen,Partner,Dependents,Phone Service,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Total Charges,Churn Label,Churn Reason
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043,7043,1869
unique,7043,1,1,1129,1652,2,2,2,2,2,...,3,3,3,3,3,2,4,6531,2,20
top,3668-QPYBK,United States,California,Los Angeles,"34.159534, -116.425984",Male,No,No,No,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,,No,Attitude of support person
freq,1,7043,7043,305,5,3555,5901,3641,5416,6361,...,3095,3473,2810,2785,3875,4171,2365,11,5174,192


### Observações — `df.describe()`

#### Variáveis Numéricas

| Coluna | Achado | Ação |
|--------|--------|------|
| `Count` | Constante (`std = 0`, sempre = 1) | **Remover** |
| `Tenure Months` | Mínimo = 0 — clientes com zero meses de contrato | **Investigar** |
| `Monthly Charges` | Range razoável: R$18 a R$119, média ~R$65 | Manter |
| `Churn Value` | Média = 0.265 → **~26,5% de churn** | Target desbalanceado, requer atenção no treino |
| `Churn Score` / `CLTV` | Derivados do churn conhecido | **Remover** (data leakage) |

#### Variáveis Categóricas

| Coluna | Achado | Ação |
|--------|--------|------|
| `Country` | `unique = 1` — só "United States" | **Remover** (sem variância) |
| `State` | `unique = 1` — só "California" | **Remover** (sem variância) |
| `CustomerID` | `unique = 7043` — todos distintos | **Remover** (identificador) |
| `Total Charges` | `top = ""` com `freq = 11` | **Corrigir** (11 strings vazias) |
| `Churn Reason` | 20 categorias únicas, só em clientes que saíram | **Remover** (data leakage) |
| `Churn Label` | Duplicata de `Churn Value` | **Remover** (redundante) |

#### Conclusão

O dataset está **razoavelmente limpo**, mas exige três correções antes da modelagem:
1. Converter `Total Charges` para `float` e tratar as 11 strings vazias
2. Remover colunas com leakage, constantes e identificadores
3. Investigar os registros com `Tenure Months == 0`